# RP-T11 The Empty Shelf: oracle arm (constructed ground truth)
Support-absence is **true by construction** (paired exemplars ablated from a real-text corpus), the
support-to-failure linkage is **programmable**, and every estimator is validated against closed
forms, including the zero-linkage control in which support signals must show no value.

In [ ]:
import os, json, time, hashlib, random, math
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy import stats as st

RESULTS_DIR = os.environ.get("RPT11_RESULTS", os.path.abspath("./rpt11_results"))
os.makedirs(os.path.join(RESULTS_DIR, "figures"), exist_ok=True)
SEED = int(os.environ.get("RPT11_SEED", "11"))
rng = np.random.default_rng(SEED); random.seed(SEED)
def sha(o): return hashlib.sha256(json.dumps(o, sort_keys=True, default=str).encode()).hexdigest()
print("results ->", RESULTS_DIR)

In [ ]:
# ---- real-text universe: HumanEval+ tasks; corpus = paired exemplars; ablation = constructed OOD ----
from evalplus.data import get_human_eval_plus
DATA = get_human_eval_plus()
TASKS = [{"id": k, "text": v["prompt"], "ex": v["prompt"] + v["canonical_solution"]}
         for k, v in DATA.items()]
rng.shuffle(TASKS)
N = len(TASKS)
SUPPORTED = set(t["id"] for t in TASKS[: N // 2])          # by construction: exemplar kept in corpus
CORPUS = [t for t in TASKS if t["id"] in SUPPORTED]
# distractor exemplars so unsupported tasks still retrieve *something*
CORPUS_TEXTS = [t["ex"] for t in CORPUS]
def toks(s): return set(s.lower().split())
def support_signals(task, k=5):
    q = toks(task["text"])
    sims = sorted((len(q & toks(x)) / (len(q | toks(x)) + 1) for x in CORPUS_TEXTS), reverse=True)[:k]
    floor_s = 0.32
    return {"max_sim": sims[0], "k_eff": sum(s >= floor_s for s in sims),
            "score_mass": sum(sims), "margin": sims[0] - sims[-1],
            "oov": 1 - len(q & set().union(*[toks(x) for x in CORPUS_TEXTS[:60]])) / max(len(q), 1)}
SIG = pd.DataFrame([{**{"id": t["id"], "z": int(t["id"] not in SUPPORTED)}, **support_signals(t)}
                    for t in TASKS])
print(f"universe {N} | supported {len(SUPPORTED)} | OOD-by-construction {N - len(SUPPORTED)}")
print(SIG.groupby("z")[["max_sim", "k_eff", "score_mass"]].mean().round(3))

In [ ]:
# ---- programmable failure model: the linkage (q1, q0) is INJECTED ----
# Per-task difficulty heterogeneity plus a stratum shift: WITHOUT within-stratum variation,
# confidence signals have nothing to detect inside a stratum, complementarity is impossible by
# construction, and the two-point oracle degenerates under AUROC ties (bring-up caught all three).
R_LABEL = 9                  # replicate budget from the purity bound (p=0.3 -> leakage 0.040)
# supported-but-hard tasks MUST exist: they are the support floor's substance (first draft capped
# difficulty so low that support implied solvability, contradicting the thesis being tested)
BASE_DIFF = {t["id"]: float(np.clip(rng.uniform(0.05, 0.80), 0, 1)) for t in TASKS}
LINK = 0.35                  # injected stratum shift: the support-to-failure linkage
def true_fail_prob(tid):
    return float(np.clip(BASE_DIFF[tid] + LINK * (tid not in SUPPORTED), 0.02, 0.95))
Q1 = float(np.mean([true_fail_prob(t["id"]) for t in TASKS if t["id"] not in SUPPORTED]))
Q0 = float(np.mean([true_fail_prob(t["id"]) for t in TASKS if t["id"] in SUPPORTED]))
def outcome_labels(R, seed):
    r2 = np.random.default_rng(seed)
    # y_label = fails ALL R attempts; per-attempt pass prob = 1 - true_fail_prob (Bernoulli attempts)
    return {t["id"]: int(r2.binomial(R, 1 - true_fail_prob(t["id"])) == 0) for t in TASKS}
Y = outcome_labels(R_LABEL, SEED + 1)
SIG["y"] = SIG["id"].map(Y)
print(f"injected linkage q1={Q1} q0={Q0} | label replicates R={R_LABEL}")
print(f"measured fail rate by stratum: {SIG.groupby('z')['y'].mean().round(3).to_dict()}")

In [ ]:
# ---- G0a: Proposition 1 identification: predicted flagging behavior from (u,v,q1,q0) vs measured ----
thr = float(np.quantile(SIG.max_sim, 0.5))
D = (SIG.max_sim < thr).astype(int)                       # detector: low similarity -> flag OOD
u = float(((D == 1) & (SIG.z == 1)).sum() / (SIG.z == 1).sum())   # TPR vs constructed z
v = float(((D == 1) & (SIG.z == 0)).sum() / (SIG.z == 0).sum())   # FPR vs constructed z
pi_z = SIG.z.mean()
# Prop-1 identity: precision of failure-flagging = P(y|flag) from total probability
pred_prec = (u * pi_z * Q1 + v * (1 - pi_z) * Q0) / (u * pi_z + v * (1 - pi_z))
meas_prec = float(SIG[D == 1].y.mean())
# label-noise correction: observed y uses R attempts, so effective rates are q^R-ish leakage-adjusted
q1_eff = float(SIG[SIG.z == 1].y.mean()); q0_eff = float(SIG[SIG.z == 0].y.mean())
pred_prec_eff = (u * pi_z * q1_eff + v * (1 - pi_z) * q0_eff) / (u * pi_z + v * (1 - pi_z))
ok_ident = abs(meas_prec - pred_prec_eff) < 0.06
print(f"detector vs constructed z: TPR {u:.2f} FPR {v:.2f}")
print(f"Prop-1 identity: measured flag failure-precision {meas_prec:.3f} vs predicted {pred_prec_eff:.3f} "
      f"(from u,v and measured conditional rates) -> ok={ok_ident}")

# ---- G0b: ZERO-LINKAGE control: same signals, failure independent of support ----
zr = np.random.default_rng(SEED + 2)
y0 = pd.Series(zr.binomial(1, 0.35, size=N), index=SIG.index)   # failure independent of support
auc0 = st.mannwhitneyu(SIG[y0 == 1].max_sim, SIG[y0 == 0].max_sim).statistic / \
       max(((y0 == 1).sum() * (y0 == 0).sum()), 1)
ok_zero = abs(auc0 - 0.5) < 0.07
print(f"zero-linkage control: support-signal AUROC vs failure = {auc0:.3f} (must be ~0.5) ok={ok_zero}")
G0 = ok_ident and ok_zero
print(f"\nGATE G0: {'PASS' if G0 else 'FAIL'}")

In [ ]:
# ---- E1: risk-coverage: support signals vs confidence baseline vs oracle; the support floor ----
def auroc(score, y):
    a, b = score[y == 1], score[y == 0]
    if len(a) == 0 or len(b) == 0: return 0.5
    return st.mannwhitneyu(a, b).statistic / (len(a) * len(b))
# confidence baseline: noisy view of true pass prob (post-generation signal analogue, has gen cost)
conf_fail = np.array([true_fail_prob(t["id"]) for t in TASKS]) + rng.normal(0, 0.13, N)
support_score = -SIG.max_sim.values                      # low similarity -> predicts failure
y = SIG.y.values
res = {"support(max_sim)": auroc(support_score, y),
       "support(k_eff)":   auroc(-SIG.k_eff.values.astype(float), y),
       "support(score_mass)": auroc(-SIG.score_mass.values, y),
       "confidence(noisy p)": auroc(conf_fail, y),
       "oracle(true q)":    auroc(np.array([true_fail_prob(t["id"]) for t in TASKS]), y)}
for k_, v_ in res.items(): print(f"  AUROC vs failure: {k_:22} {v_:.3f}")
frac = (res["support(max_sim)"] - 0.5) / max(res["oracle(true q)"] - 0.5, 1e-9)
print(f"E-H1: support signal recovers {frac:.0%} of oracle abstention value (zero generation cost)")

# risk-coverage curves + the support floor
def risk_cov(score):
    order = np.argsort(-score)                            # abstain on worst first -> keep best
    ys = y[order]
    cov = np.arange(1, N + 1) / N
    risk = np.cumsum(ys[::-1])[::-1] / np.arange(N, 0, -1)  # selective risk among kept... simpler:
    kept_risk = [ys[i:].mean() for i in range(N)]
    return cov[::-1], np.array(kept_risk)
cov_s, risk_s = risk_cov(support_score)
floor_q0 = SIG[SIG.z == 0].y.mean()
hi = risk_s[np.searchsorted(cov_s, 0.5)]
print(f"support floor: selective risk at coverage ~ supported mass "
      f"{hi:.3f} >= q0_eff {floor_q0:.3f}: {hi >= floor_q0 - 0.03}")

In [ ]:
# ---- E2: complementarity by stratum (E-H2) ----
sup_mask = SIG.z.values == 0
# Stratified analysis runs on R=3 labels: R=9 purity leaves the supported stratum without failure
# labels entirely (a real design tension the calculator must expose: purity trades against
# stratified power). Purity caveat printed with the result.
Y3 = outcome_labels(3, SEED + 77)
y3 = SIG["id"].map(Y3).values
print(f"stratified labels at R=3: supported fail rate {y3[sup_mask].mean():.3f}, "
      f"unsupported {y3[~sup_mask].mean():.3f}  (R=9 supported rate was ~0)")
rows = []
for name, sc in [("support", support_score), ("confidence", conf_fail)]:
    rows.append({"signal": name,
                 "AUROC_unsupported": auroc(sc[~sup_mask], y3[~sup_mask]),
                 "AUROC_supported":   auroc(sc[sup_mask], y3[sup_mask]),
                 "AUROC_overall":     auroc(sc, y3)})
comp = pd.DataFrame(rows); print(comp.round(3).to_string(index=False))
# composition
combo = 0.5 * (st.rankdata(support_score) / N) + 0.5 * (st.rankdata(conf_fail) / N)
a_combo = auroc(combo, y3)
eh2 = (comp.iloc[1].AUROC_supported > comp.iloc[0].AUROC_supported) and \
      (a_combo >= max(comp.AUROC_overall) - 0.01)
print(f"composed AUROC {a_combo:.3f} | E-H2 stratified complementarity: {eh2}")
comp.to_csv(os.path.join(RESULTS_DIR, "complementarity.csv"), index=False)

In [ ]:
# ---- E3: certification budgets and label purity (Prop 2, empirical) ----
# (a) CI width at benchmark scale matches the binomial formula
flag = D == 1; n_flag = int(flag.sum())
p_hat = float(SIG[flag].y.mean())
w_meas = 1.96 * math.sqrt(p_hat * (1 - p_hat) / n_flag)
print(f"certification: n_flag={n_flag} -> precision half-width {w_meas:.3f} "
      f"(the outline's 164-task/10%-flag case predicts ~0.15)")
# (b) label purity: leakage of p=0.3-pass tasks into the failure class vs R, against (1-p)^R
purity = []
for R in [1, 3, 5, 9]:
    yR = outcome_labels(R, SEED + 40 + R)
    leak = np.mean([yR[t["id"]] for t in TASKS if t["id"] in SUPPORTED])
    theory = np.mean([true_fail_prob(t["id"]) ** R for t in TASKS if t["id"] in SUPPORTED])
    purity.append({"R": R, "measured_leakage": leak, "theory": float(theory)})
PUR = pd.DataFrame(purity)
PUR["ok"] = (PUR.measured_leakage - PUR.theory).abs() < 0.05
print(PUR.round(4).to_string(index=False))
eh4 = bool(PUR.ok.all())
PUR.to_csv(os.path.join(RESULTS_DIR, "label_purity.csv"), index=False)
print(f"E-H4 purity follows (1-p)^R within tolerance: {eh4}")

In [ ]:
# ---- adjudication + export ----
ADJ = pd.DataFrame([
 {"claim": "G0a Prop-1 identity: predicted flag precision matches measured", "verdict": "PASS" if ok_ident else "FAIL"},
 {"claim": "G0b zero-linkage control: support signals show no value", "verdict": "PASS" if ok_zero else "FAIL"},
 {"claim": "E-H1 support signals recover material oracle value at zero gen cost",
  "verdict": "PASS" if frac >= 0.5 else "FAIL"},
 {"claim": "E-H2 complementarity by stratum; composition not dominated", "verdict": "PASS" if eh2 else "FAIL"},
 {"claim": "E-floor selective risk bounded by q0 at high coverage", "verdict": "PASS" if hi >= floor_q0 - 0.03 else "FAIL"},
 {"claim": "E-H4 label purity follows the (1-p)^R correction", "verdict": "PASS" if eh4 else "FAIL"},
])
print(ADJ.to_string(index=False))
ADJ.to_csv(os.path.join(RESULTS_DIR, "adjudication_oracle.csv"), index=False)
SIG.to_csv(os.path.join(RESULTS_DIR, "signals.csv"), index=False)
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].plot(cov_s, risk_s, label="support-only abstention")
ax[0].axhline(floor_q0, ls="--", color="#a33", label="support floor (q0)")
ax[0].set_xlabel("coverage"); ax[0].set_ylabel("selective risk"); ax[0].legend(fontsize=8)
ax[0].set_title("The Empty Shelf: risk-coverage and the floor")
x = np.arange(len(res)); ax[1].bar(x, [v - 0.5 for v in res.values()])
ax[1].set_xticks(x); ax[1].set_xticklabels(res.keys(), rotation=25, fontsize=7)
ax[1].set_ylabel("AUROC - 0.5"); ax[1].set_title("Signal value vs verified failure")
plt.tight_layout(); plt.savefig(os.path.join(RESULTS_DIR, "figures", "figA_shelf.png"), dpi=140)
summary = {"study": "RP-T11 oracle arm", "G0": "PASS" if G0 else "FAIL",
           "injected": {"q1": Q1, "q0": Q0, "R_label": R_LABEL},
           "detector_ops": {"u": u, "v": v}, "oracle_fraction": round(float(frac), 3),
           "aurocs": {k: round(v, 3) for k, v in res.items()},
           "adjudication": ADJ.to_dict("records")}
json.dump(summary, open(os.path.join(RESULTS_DIR, "summary.json"), "w"), indent=1)
print("\nwrote:", sorted(os.listdir(RESULTS_DIR)))